In [4]:
import os
import sys
import pandas as pd
import yaml 
from matplotlib import pyplot as plt
from matplotlib import ticker as mticker
from matplotlib import colors as mcolors
from matplotlib import patches as mpatches
import statsmodels.api as sm
import numpy as np
from itertools import product
import subprocess

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11
#plt.rcParams['text.usetex'] = True

from sklearn.decomposition import PCA

with open("../../config.yaml.local", "r") as f:
    LOCAL_CONFIG = yaml.safe_load(f)
with open("../../config.yaml", "r") as f:
    CONFIG = yaml.safe_load(f)
sys.path.append("../python")

import globals
import data_tools as dt
import utils
import emb

LOCAL_PATH = LOCAL_CONFIG["LOCAL_PATH"]
RAW_DATA_PATH = LOCAL_CONFIG["RAW_DATA_PATH"]
DATA_PATH = LOCAL_CONFIG["DATA_PATH"]
R_PATH = LOCAL_CONFIG["R_PATH"]

RUN_R_SCRIPTS = False
REPLACE_ANALYSIS_DATA = False

analysis_data_filepath = os.path.join(
    DATA_PATH, "v4v-analysis-2.parquet"
)

TITLE_PCA_K = 10
TEXT_PCA_K = 10


In [5]:
# Making analysis data for R

if not os.path.exists(analysis_data_filepath) or REPLACE_ANALYSIS_DATA:

    df = dt.get_post_quality_analysis_data()
    df = df.loc[df['title'] != 'deleted by author'].reset_index(drop=True)

    # Dynamically process embeddings instead of loading from
    # post_embeddings.pkl to save on memory
    title_embeddings = []
    text_embeddings = []

    for idx, row in df.iterrows():
        title_emb = np.array(emb.get_embedding_robust(row['title']))
        title_emb = title_emb / np.linalg.norm(title_emb)
        title_embeddings.append(title_emb)
        
        text_emb = np.array(emb.get_embedding_robust(row['text']))
        text_emb = text_emb / np.linalg.norm(text_emb)
        text_embeddings.append(text_emb)

    title_embeddings = np.array(title_embeddings)
    text_embeddings = np.array(text_embeddings)    

    # scree plot for title embeddings
    title_pca = PCA()
    title_pca.fit(title_embeddings)
    explained_variance = title_pca.explained_variance_ratio_[0:50]
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker='o')
    #plt.title('Title Embeddings: PCA Scree Plot')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance Ratio')
    plt.grid()
    filename = os.path.join(LOCAL_PATH, 'results', 'fig_title_scree_plot.pdf')
    plt.savefig(filename, bbox_inches='tight')
    plt.show()

    # scree plot for text embeddings
    text_pca = PCA()
    text_pca.fit(text_embeddings)
    explained_variance = text_pca.explained_variance_ratio_[0:50]
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker='o')
    #plt.title('Text Embeddings: PCA Scree Plot')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance Ratio')
    plt.grid()
    filename = os.path.join(LOCAL_PATH, 'results', 'fig_text_scree_plot.pdf')
    plt.savefig(filename, bbox_inches='tight')
    plt.show()

    # PCA for embeddings
    title_pca = PCA(n_components=TITLE_PCA_K)
    title_pca.fit(title_embeddings)
    title_pca_embeddings = title_pca.transform(title_embeddings)
    for k in range(TITLE_PCA_K):
        df[f'title_emb_{k}'] = title_pca_embeddings[:, k]

    text_pca = PCA(n_components=TEXT_PCA_K)
    text_pca.fit(text_embeddings)
    text_pca_embeddings = text_pca.transform(text_embeddings)
    for k in range(TEXT_PCA_K):
        df[f'text_emb_{k}'] = text_pca_embeddings[:, k]

    # Output analysis data file
    df.to_parquet(analysis_data_filepath, index=False)
    print(f"Saved analysis data to {analysis_data_filepath}")
else:
    df = pd.read_parquet(analysis_data_filepath)
    print(f"Loaded analysis data from {analysis_data_filepath}")



Loaded analysis data from C:/Users/edwar/Dropbox/projects/sn-research/processed_data\v4v-analysis-2.parquet


In [6]:
# Running initial R regression to get quality measures

res = subprocess.run([R_PATH, LOCAL_PATH + "/src/R/v4v-analysis-2.R"], check=True, capture_output=True, text=True)
print(res.stdout)

                                 r1                 r2                  r3
Dependent Var.:          log_sats48         log_sats48          log_sats48
                                                                          
Constant          3.253*** (0.3137)  3.200*** (0.3410)                    
log_numwords     0.3222*** (0.0341)   0.2543* (0.1010)  0.1614*** (0.0440)
is_link_postTRUE  -0.4845* (0.2389)  -0.1969* (0.0944) -0.4041*** (0.0578)
num_img_or_links   0.0199. (0.0117)   0.0153* (0.0071)    0.0107. (0.0058)
text_emb_0                             0.4629 (0.3735)     0.0824 (0.1067)
text_emb_1                             0.5504 (0.5270)    -0.1798 (0.1833)
text_emb_2                            -0.4269 (0.4429)   0.5939** (0.2117)
text_emb_3                            -0.4908 (0.7180)    0.5333* (0.2319)
text_emb_4                           -1.650** (0.5576)   -0.4437* (0.1747)
text_emb_5                          -2.673*** (0.3640)  -1.823*** (0.1912)
text_emb_6               

In [12]:
# Zaps regression table

coefs_df = pd.read_parquet(os.path.join(DATA_PATH, "v4v-analysis-2-zreg.parquet"))

header = r"""\begin{table}[H]
\centering
\caption{Regression of log(Zaps) on Post Attributes} \label{tbl_v4v_zreg}
\begin{threeparttable}
\begin{tabular}{@{\extracolsep{5pt}}lccc} 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{3}{c}{log(Zaps)} \\ 
\\[-1.8ex] & (1) & (2) & (3) \\ 
\hline \\[-1.8ex] 
"""
footer = r"""\end{tabular} 
\begin{tablenotes}[flushleft]\footnotesize
\item[] \parbox[t]{\linewidth}{%
\hfill * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
}
\item \textit{Notes:} Regression of number of sats a post earns in the first 48 hours on post attributes. The unit of observation is a post. Standard errors are clustered by territory.
\end{tablenotes}
\end{threeparttable}
\end{table}
"""
reg_names = ["r1", "r2", "r3"]

vars = [
    ("log_numwords", "log(\\# Words)"),
    ("is_link_postTRUE", "Is link post"),
    ("num_img_or_links", "\\# Images and links in post body"),
    ("Constant", "(Intercept)"),
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\" + "\n"

tbl += r"""& & & & \\
\hline \\ [-1.8ex]
Text Embeddings         & N & Y & Y \\
Territory FE            & N & N & Y \\
User FE                 & N & N & Y \\
"""

tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\" + "\n"

tbl += "$R^2$ "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="R2")
    r2 = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {r2:.3f}"
tbl += r" \\ \hline \hline " + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "results", "tbl_v4v_zreg.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)

\begin{table}[H]
\centering
\caption{Regression of log(Zaps) on Post Attributes} \label{tbl_v4v_zreg}
\begin{threeparttable}
\begin{tabular}{@{\extracolsep{5pt}}lccc} 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{3}{c}{log(Zaps)} \\ 
\\[-1.8ex] & (1) & (2) & (3) \\ 
\hline \\[-1.8ex] 
log(\# Words)  & 0.322$^{***}$ & 0.254$^{**}$ & 0.161$^{***}$ \\
 & (0.034) & (0.101) & (0.044) \\
Is link post  & -0.485$^{**}$ & -0.197$^{**}$ & -0.404$^{***}$ \\
 & (0.239) & (0.094) & (0.058) \\
\# Images and links in post body  & 0.020$^{*}$ & 0.015$^{**}$ & 0.011$^{*}$ \\
 & (0.012) & (0.007) & (0.006) \\
(Intercept)  &  &  &  \\
 &  &  &  \\
& & & & \\
\hline \\ [-1.8ex]
Text Embeddings         & N & Y & Y \\
Territory FE            & N & N & Y \\
User FE                 & N & N & Y \\
Observations  & 175,857 & 175,857 & 174,578 \\
$R^2$  & 0.155 & 0.180 & 0.404 \\ \hline \hline 
\end{tabular} 
\begin{tablenotes}

In [15]:
# Zaps regression table

coefs_df = pd.read_parquet(os.path.join(DATA_PATH, "v4v-analysis-2-qreg.parquet"))

header = r"""\begin{table}[H]
\centering
\caption{Regression of Next Post Quality on Past Zap Surprises} \label{tbl_v4v_qreg}
\begin{threeparttable}
\begin{tabular}{@{\extracolsep{5pt}}lccc} 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{3}{c}{$q_i$ of next post} \\ 
\\[-1.8ex] & (1) & (2) & (3) \\ 
\hline \\[-1.8ex] 
"""
footer = r"""\end{tabular} 
\begin{tablenotes}[flushleft]\footnotesize
\item[] \parbox[t]{\linewidth}{%
\hfill * $p<0.1$, ** $p<0.05$, *** $p<0.01$.
}
\item \textit{Notes:} Regression of next post quality ($q_i$) on the average of prior zap surprises ($\frac{1}{n} \sum \ln z_i - q_i$). The unit of observation is a post. Standard errors are clustered by territory.
\end{tablenotes}
\end{threeparttable}
\end{table}
"""
reg_names = ["r4", "r5", "r6"]

vars = [
    ("avg_surprise", r"Avg of past zap surprises ($\hat{\theta})$"),
    ("log_prior_zaps", "log(Total prior zaps)"),
    ("log_prior_posts", "log(Total prior posts)")
]

tbl = ""
for v in vars:
    tbl += v[1] + " "
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        coef = coefs_df.loc[idx, "estimate"].values[0]
        serr = coefs_df.loc[idx, "serr"].values[0]
        stars = utils.stars(coef, serr)
        tbl += f" & {coef:.3f}$^{{{stars}}}$"
    tbl += r" \\" + "\n"
    for rn in reg_names:
        idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]==v[0])
        if idx.sum()==0:
            tbl += " & "
            continue
        serr = coefs_df.loc[idx, "serr"].values[0]
        tbl += f" & ({serr:.3f})"
    tbl += r" \\" + "\n"

tbl += r"""& & & & \\
\hline \\ [-1.8ex]
Territory FE            & Y & Y & Y \\
User FE                 & Y & Y & Y \\
Week FE                 & N & N & Y \\
"""

tbl += "Observations "
for rn in reg_names:
    idx = (coefs_df["regression_name"]==rn) & (coefs_df["coef_name"]=="num_obs")
    nobs = coefs_df.loc[idx, "estimate"].values[0]
    tbl += f" & {nobs:,.0f}"
tbl += r" \\ \hline \hline " + "\n"

table_tex = header + tbl + footer    

with open(os.path.join(LOCAL_PATH, "results", "tbl_v4v_qreg.tex"), "w") as f:
    f.write(table_tex)

print(table_tex)

\begin{table}[H]
\centering
\caption{Regression of Next Post Quality on Past Zap Surprises} \label{tbl_v4v_qreg}
\begin{threeparttable}
\begin{tabular}{@{\extracolsep{5pt}}lccc} 
\\[-1.8ex]\hline 
\hline \\[-1.8ex] 
 & \multicolumn{3}{c}{\textit{Dependent variable:}} \\ 
\cline{2-4} 
\\[-1.8ex] & \multicolumn{3}{c}{$q_i$ of next post} \\ 
\\[-1.8ex] & (1) & (2) & (3) \\ 
\hline \\[-1.8ex] 
Avg of past zap surprises ($\hat{\theta})$  & 0.048$^{***}$ & 0.034$^{***}$ & 0.024$^{***}$ \\
 & (0.007) & (0.008) & (0.006) \\
log(Total prior zaps)  &  & 0.016$^{***}$ & -0.012$^{***}$ \\
 &  & (0.004) & (0.003) \\
log(Total prior posts)  &  & 0.000$^{***}$ & -0.000$^{}$ \\
 &  & (0.000) & (0.000) \\
& & & & \\
\hline \\ [-1.8ex]
Territory FE            & Y & Y & Y \\
User FE                 & Y & Y & Y \\
Week FE                 & N & N & Y \\
Observations  & 170,936 & 170,936 & 170,936 \\ \hline \hline 
\end{tabular} 
\begin{tablenotes}[flushleft]\footnotesize
\item[] \parbox[t]{\linewidth}{%
\h